# Dataset Evaluation Example

이 Notebook은 **공통 평가 모듈을 호출하고, 기관별 Parser/Converter/Model 연결 코드는 Notebook에서 직접 작성**하는 예제입니다.

- `src/`: 공통 제공 모듈 → 가능하면 수정하지 않음
- Notebook: 기관별 데이터/모델에 맞는 Parser·Converter·Model 연결


In [1]:
import os
 
from evaluator.data_loader import build_detection_dataset
from evaluator.parser import YOLOParser
from evaluator.model_utils import load_detection_model
from evaluator.evaluator import evaluate_detection_model, print_evaluation_result

I0000 00:00:1789437525.684035  917074 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789437525.744483  917074 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789437527.384629  917074 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 1. 기관별 설정

기관은 자신의 Croissant metadata와 모델 경로만 지정합니다.

In [2]:
JSONLD_PATH = 'MAX_TR_DS01.jsonld'
MODEL_PATH = 'yolov8_full.keras'

SPLIT = 'test'
IMG_SIZE = (640, 640)
MAX_BOXES = 2
BATCH_SIZE = 32

## 2. Annotation Parser

현재 예제 Dataset은 YOLO Annotation을 사용하므로 공통으로 제공되는 `YOLOParser`를 사용합니다.

기관에서 별도 Annotation 형식을 사용하는 경우 **이 Notebook에서 직접 Parser를 구현**할 수 있습니다.

In [3]:
parser = YOLOParser(max_boxes=MAX_BOXES)

# 예: 기관 자체 Annotation 형식이라면 Notebook에서 직접 구현
# class MyParser:
#     def parse(self, annotation):
#         ...
#         return boxes, classes, num_boxes
#
# parser = MyParser()

## 3. Croissant Dataset Load

`data_loader.py`는 Croissant Dataset을 읽고 Annotation 처리를 `parser`에 위임합니다.

In [5]:
dataset = build_detection_dataset(
    jsonld_path=JSONLD_PATH,
    split=SPLIT,
    img_size=IMG_SIZE,
    max_boxes=MAX_BOXES,
    batch_size=BATCH_SIZE,
    parser=parser,
)

print('element_spec:', dataset.element_spec)
for images, boxes, classes, num_boxes in dataset.take(1):
    print('첫 배치 이미지:', tuple(images.shape))
    print('첫 배치 박스  :', tuple(boxes.shape))
    print('첫 배치 클래스:', tuple(classes.shape))
    print('유효 박스 수  :', num_boxes.numpy().tolist())


element_spec: (TensorSpec(shape=(None, 640, 640, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 2, 4), dtype=tf.float32, name=None), TensorSpec(shape=(None, 2), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))
첫 배치 이미지: (32, 640, 640, 3)
첫 배치 박스  : (32, 2, 4)
첫 배치 클래스: (32, 2)
유효 박스 수  : [2, 2, 2, 1, 2, 2, 2, 1, 1, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 4. 기관별 Model Input Converter

기관은 자신의 모델 입력 규격에 맞는 변환 함수를 이 Notebook에서 직접 작성합니다.

아래는 YOLO의 normalized `[cx, cy, w, h]`를 KerasCV의 pixel `[x1, y1, x2, y2]`로 변환하는 예시입니다.

In [6]:
import tensorflow as tf

def convert_for_my_model(dataset):
    @tf.autograph.experimental.do_not_convert
    def _convert(images, boxes, classes, num_boxes):
        cx, cy, w, h = tf.unstack(boxes, axis=-1)

        x1 = (cx - w / 2.0) * IMG_SIZE[1]
        y1 = (cy - h / 2.0) * IMG_SIZE[0]
        x2 = (cx + w / 2.0) * IMG_SIZE[1]
        y2 = (cy + h / 2.0) * IMG_SIZE[0]

        xyxy = tf.stack([x1, y1, x2, y2], axis=-1)

        return images, {
            'boxes': xyxy,
            'classes': classes,
        }

    return dataset.map(_convert)


test_ds = convert_for_my_model(dataset)

### 기관별 Converter 작성 예시

모델이 KerasCV가 아닌 경우에도 동일한 위치에서 필요한 형태로 변환하면 됩니다.

In [7]:
# 예시
# def convert_for_my_model(dataset):
#     def _convert(images, boxes, classes, num_boxes):
#         # 기관 자체 모델의 입력 규격에 맞게 변환
#         ...
#         return images, model_inputs
#     return dataset.map(_convert)
#
# test_ds = convert_for_my_model(dataset)


## 5. Model Load

In [9]:
model = load_detection_model(
    MODEL_PATH,
    confidence_threshold=0.01,
    iou_threshold=0.5,
)

## 6. Evaluation

모델과 평가 모듈은 공통 기능을 사용합니다.

In [11]:
res = evaluate_detection_model(model, test_ds)
print_evaluation_result(res, split=SPLIT)

=== 평가 결과 (test셋) ===
  mAP50      (IoU=0.5)   : 0.0834
  mAP50:95   (COCO 기본) : 0.0539
  Recall@100             : 0.4313
